# imports

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

# from xmip.utils import google_cmip_col
from xmip.preprocessing import combined_preprocessing
from xmip.postprocessing import concat_experiments, merge_variables
import xarray as xr
import numpy as np
import xesmf as xe
import pandas as pd
import matplotlib.pyplot as plt
from xmip.utils import cmip6_dataset_id
import intake
from dask.diagnostics import ProgressBar
import gcsfs
fs = gcsfs.GCSFileSystem()

# functions needed for processing

In [ ]:
def replace_calendar(ds:xr.Dataset) -> xr.Dataset:
    """
    Replaces dates in ESM output for consistency across all ESM members.

    Arguments:
        ds(xr.Dataset): ESM output in the form of an Xarray Dataset, with a time dimension.

    Returns:
        ds(xr.Dataset): ESM output in the form of an Xarray Dataset, with adjusted dates.
    """
    year = ds.time.data[0].year
    month = ds.time.data[0].month
    start_date = f'{year}-{month:0>2}-01'
    new_monthly_time = xr.cftime_range(start_date, periods=len(ds.time), freq='1MS')# + np.timedelta64(14, 'D')
    ds = ds.assign_coords(time=new_monthly_time)
    return ds

In [ ]:
def regrid(target_grid, ds:xr.Dataset) -> xr.Dataset:
    """
    Regrids ESM output to chosen lat x lon grid. 
    E.g. longitude between -180 to 180 degrees, vs 0 to 360 degrees.

    Arguments:
        target_grid(xr.Dataset): Xarray Dataset of intended time/lat/lon frequencies/bounds.
        
        ds(xr.Dataset): ESM output in the form of an Xarray Dataset.

    Returns:
         ds(xr.Dataset): ESM output in the form of an Xarray Dataset, 
             moved to target_grid bounds.
    """
    
    regridder = xe.Regridder(ds, target_grid, 'bilinear', ignore_degenerate=True, periodic=True) 
    ds_regridded = regridder(ds, keep_attrs=True)
    
    return ds_regridded

In [ ]:
def full_testbed_processing(target_grid, init_year, fin_year, ds:xr.Dataset) -> xr.Dataset:
    """
    Runs "regrid" and "replace_calendar" functions on ESM output.

    Arguments:
        target_grid(xr.Dataset): Xarray Dataset of intended time/lat/lon frequencies/bounds.
        
        init_year(int): First year of ESM output.
        
        fin_year(int): Final year of ESM output.
        
        ds(xr.Dataset): ESM output in the form of an Xarray Dataset.

    Returns:
        ds_new_cal(xr.Dataset): ESM output with new dates/times and adjusted to ideal grid.
    """
    
    ds = ds.squeeze(drop=True)
    if 'lev' in ds.dims:
        ds = ds.isel(lev=0).drop_vars(('lev'))
    
    ds = ds.sel(time=slice(str(init_year),str(fin_year)))
    
    assert len(ds.time) == 3012
    assert ds.time.data[0].year == 1850
    
    # Processing
    ds_regridded = regrid(target_grid, ds)
    ds_new_cal = replace_calendar(ds_regridded)

    return ds_new_cal

# getting/processing ESM output (SSP245 and SSP585)
SSP245 and SSP585 refer to mid and high emissions scenarios, respectively.

In [ ]:
# filter the full CMIP6 catalog for data we could use
url = "https://storage.googleapis.com/cmip6/cmip6-pgf-ingestion-test/catalog/catalog.json" # Only stores that pass current tests
col = intake.open_esm_datastore(url)

# where to save processed ESM output; insert leap-persistent username
save_dir = 'gs://leap-persistent/{INSERT_USERNAME}/phytoplankton/gridded_data_1850-2100'

# surface ocean (SO) temperature, SO salinity, SO pco2, SO pH, Interior Primary Organic Carbon Production
variables = ['tos', 'sos', 'spco2', 'phos', 'intpp']

In [ ]:
def get_esm_output(ssp_no, save_dir, variables):
    """
    Searches for ESM output of needed variables for SSP emissions scenario specified. Then processes (e.g. regrids) and saves output in Xr.Datasets.

    Arguments:
        ssp_no(str): SSP number (emissions scenario).
        
        variables(list): List of variables wanted from ESMs. Each variable must be a string, in CMIP6 abbreviated format (e.g. 'tos' = surface ocean temperature). 
            Check CMIP6 documentation for proper abbreviations.
    """

    ### search for data ###
    esm_catalog = col.search(
        variable_id = variables,
        table_id = ['Omon'], # monthly ocean output only
        experiment_id = ['historical', ssp_no],
        require_all_on = ['source_id', 'member_id', 'grid_label'] # this ensures that results will have all variables and experiments available
    )
    no_esms = len(esm_catalog.df.groupby(['source_id', 'grid_label'])[['member_id']].nunique())
    no_mems = esm_catalog.df.groupby(['source_id', 'grid_label'])[['member_id']].nunique().sum().values[0]
    
    print(f'Available ESMs and # of members: {no_esms} ESMs, {no_mems} members')

    ##############################################

    ### turn data into dataset dictionary ###
    esm_dict = esm_catalog.to_dataset_dict(
        preprocess=combined_preprocessing,
        xarray_open_kwargs=dict(use_cftime=True),
        aggregate=False
    )

    ##############################################

    ### fix time bugs  ###
    # necessary due to bug in some ESM output time data, hard-coded to ssp245/ssp585 #
    
    if ssp_no == 'ssp245':
        good_fut_time = esm_dict['ScenarioMIP.CCCma.CanESM5.ssp245.r1i1p2f1.Omon.sos.gn.none.r1i1p2f1.v20190429.gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp245/r1i1p2f1/Omon/sos/gn/v20190429/'].time

    elif ssp_no == 'ssp585':
        good_fut_time = esm_dict['ScenarioMIP.CCCma.CanESM5.ssp585.r1i1p2f1.Omon.sos.gn.none.r1i1p2f1.v20190429.gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp585/r1i1p2f1/Omon/sos/gn/v20190429/'].time

    good_hist_time = esm_dict['CMIP.CCCma.CanESM5.historical.r2i1p2f1.Omon.intpp.gn.none.r2i1p2f1.v20190429.gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical/r2i1p2f1/Omon/intpp/gn/v20190429/'].time

    for keys, item in esm_dict.items():
    
        if ssp_no in keys:
            item = item.sel(time=slice('2015','2100'))
            
            if len(item.time) == 1032 and item.time.dt.year.values[0] == 2015 and item.time.dt.year.values[-1] == 2100:
                item['time'] = good_fut_time
                esm_dict[keys] = item
        
            else:
                print('bad fut key:',keys)
                print(len(item.time) == 1032, item.time.dt.year.values[0] == 2015, item.time.dt.year.values[-1] == 2100)
                
        elif 'historical' in keys:
            item = item.sel(time=slice('1850','2014'))
    
            if len(item.time) == 1980 and item.time.dt.year.values[0] == 1850 and item.time.dt.year.values[-1] == 2014:
                item['time'] = good_hist_time
                esm_dict[keys] = item
                
            else:
                print('bad hist key:',keys)
                print(len(item.time) == 1980, item.time.dt.year.values[0] == 1850, item.time.dt.year.values[-1] == 2014)

    ##############################################

    ### merge all variables from each ESM member + concatenate historical with ssp ###

    esm_ds = merge_variables(esm_dict)
    esm_ds = concat_experiments(esm_ds)

    ##############################################

    ### regridding to 1x1 degree resolution, aligning all dates/times across all members ###
    
    ylat = xr.DataArray(data=[x+.5 for x in range(-90, 90, 1)], dims=['ylat'], coords=dict( ylat=(['ylat'],[x+.5 for x in range(-90, 90, 1)]) ),)
    xlon = xr.DataArray(data=[x+.5 for x in range(-180,180,1)], dims=['xlon'], coords=dict( xlon=(['xlon'],[x+.5 for x in range(-180,180,1)]) ),)
    
    init_year = 1850
    fin_year = 2100
    
    ttime = pd.date_range(start=str(str(init_year)), end=str(fin_year),freq='MS') + np.timedelta64(14, 'D') #time should be monthly on the middle of the month
    # note that the time doesnt affect regridding but we do use this time to overwrite the monthly dates so its consistent
    target_grid = xr.Dataset({'time':(['time'],ttime.values), 'latitude':(['latitude'],ylat.values),'longitude':(['longitude'],xlon.values)}) #must be named this way for old XESFM versions

    ##############################################

    ### saving processed members ###

    member_counter = 0
    for k,item in esm_ds.items():
        item_out_ssp = full_testbed_processing(target_grid, init_year, fin_year, item)
        
        item_id = cmip6_dataset_id(item_out_ssp, id_attrs=[
            'source_id',
            'variant_label',
            'table_id'
        ])
    
        vars_to_drop = ['lev_bounds','time_bounds','lev_partial','lat_verticies','lon_verticies','lon_bounds','lat_bounds']
    
        for var in vars_to_drop:
            if var in item_out_ssp:
                item_out_ssp = item_out_ssp.drop_vars(var)
        
        save_path = f"{save_dir}/{ssp_no}/{item.attrs['source_id']}/member_{item.attrs['intake_esm_attrs:member_id']}/{item_id}.zarr"
        print(f"Writing to {save_path = }")
        with ProgressBar():
            item_out_ssp.chunk({'time':200}).to_zarr(save_path, mode='w')
        member_counter +=1

    ##############################################

In [ ]:
get_esm_output('ssp245', save_dir, variables)